# Teaching a Network to Bend: Hidden Layers and Activation Functions
**Series A: Machine Learning Basics for Chemists, Post 2** &nbsp;|&nbsp; companion notebook

In Post 1 we fitted $\sin(x)$ with a cubic whose features we chose by hand. Here a **neural network chooses its own features**.
We build one from scratch in NumPy, write the backpropagation ourselves, and see what each activation function does.
Every number quoted in the post is reproduced here.

**How to use this notebook**
1. Run the cells top to bottom (**Runtime → Run all**). The whole notebook takes about a minute.
2. Before running a cell marked **Stop and predict**, guess the output.
3. Finish with the **Your turn** exercises.

Nothing needs installing. PyTorch is used only once, at the end, to confirm our hand-written gradients.

## 0. Setup: data and the network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- data: 200 points of sin(x), as in the post ----
N = 200
x = np.linspace(-np.pi, np.pi, N)
y = np.sin(x)
X = x[:, None]                                   # shape (N, 1)
x_test = np.linspace(-np.pi, np.pi, 1000)        # fresh points inside the training range
x_far  = np.linspace(-2*np.pi, 2*np.pi, 2000)    # also OUTSIDE the training range

# ---- activation functions and their derivatives ----
# Each derivative is written using the activation value a = phi(z), which backprop already has.
ACT = {
    "tanh":    (np.tanh,                        lambda z, a: 1 - a**2),
    "sigmoid": (lambda z: 1 / (1 + np.exp(-z)), lambda z, a: a * (1 - a)),
    "relu":    (lambda z: np.maximum(0, z),     lambda z, a: (z > 0).astype(float)),
    "linear":  (lambda z: z,                    lambda z, a: np.ones_like(z)),
}

# ---- a one-hidden-layer network:  yhat = sum_h v_h * phi(w_h * x + b_h) + c ----
def init(H, seed):
    rng = np.random.default_rng(seed)
    return dict(W1=rng.standard_normal((1, H)),              # w_h
                b1=rng.standard_normal(H),                   # b_h
                W2=rng.standard_normal((H, 1)) / np.sqrt(H), # v_h
                b2=np.zeros(1))                              # c

def forward(p, X, act):
    phi = ACT[act][0]
    Z = X @ p["W1"] + p["b1"]          # (N, H) pre-activations
    A = phi(Z)                         # (N, H) hidden outputs
    return Z, A, A @ p["W2"] + p["b2"] # (N, 1) predictions

def loss_and_grad(p, X, y, act):
    """Mean squared error and its gradient, by hand-written backpropagation."""
    n = len(X)
    Z, A, yhat = forward(p, X, act)
    r = yhat - y[:, None]                         # residuals
    L = np.mean(r**2)
    d2 = 2 * r / n                                # dL/dyhat
    g = {"W2": A.T @ d2, "b2": d2.sum(0)}         # output layer
    d1 = (d2 @ p["W2"].T) * ACT[act][1](Z, A)     # push the error back through phi
    g["W1"] = X.T @ d1; g["b1"] = d1.sum(0)       # hidden layer
    return L, g

def adam(p, X, y, act, steps=4000, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    m = {k: np.zeros_like(v) for k, v in p.items()}
    s = {k: np.zeros_like(v) for k, v in p.items()}
    hist = []                                      # hist[k] = loss after k updates
    for t in range(1, steps + 1):
        L, g = loss_and_grad(p, X, y, act)
        hist.append(L)
        for k in p:
            m[k] = beta1 * m[k] + (1 - beta1) * g[k]
            s[k] = beta2 * s[k] + (1 - beta2) * g[k]**2
            p[k] -= lr * (m[k] / (1 - beta1**t)) / (np.sqrt(s[k] / (1 - beta2**t)) + eps)
    hist.append(loss_and_grad(p, X, y, act)[0])
    return p, np.array(hist)

def mse(p, xx, act):                               # error against the true sine at points xx
    return float(np.mean((forward(p, xx[:, None], act)[2][:, 0] - np.sin(xx))**2))

print("setup done; a network with H hidden units has 3H + 1 =", 3*10 + 1, "parameters for H = 10")

## 1. Activation functions
A neuron computes $z=wx+b$ and passes it through $\phi$. Three common choices, and the "linear" one that does nothing.
**Stop and predict:** what is the largest value each derivative can reach? (The post says $\sigma'\le 1/4$.)

In [ ]:
z = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for name in ("sigmoid", "tanh", "relu"):
    phi, dphi = ACT[name]
    a = phi(z)
    ax[0].plot(z, a, label=name)
    ax[1].plot(z, dphi(z, a), label=name)
    print(f"{name:>8}: max derivative = {dphi(z, a).max():.3f}")
ax[0].set_title("activation  phi(z)");  ax[1].set_title("derivative  phi'(z)")
for a_ in ax: a_.grid(alpha=0.3); a_.legend(); a_.set_xlabel("z")
plt.tight_layout(); plt.show()

## 2. Why stacking linear layers is useless
With $\phi(z)=z$ the network is $\hat y=\sum_h v_h(w_hx+b_h)+c$, which is just a straight line. **Stop and predict:** after training, what slope and loss will a 10-unit *linear* network reach on $\sin(x)$?

In [ ]:
# the best straight line, found directly
slope_line, icpt_line = np.polyfit(x, y, 1)
mse_line = np.mean((slope_line * x + icpt_line - y)**2)
print(f"best line:  slope = {slope_line:.4f}, intercept = {icpt_line:+.4f}, MSE = {mse_line:.4f}")

# a 10-unit "network" with linear activations
p, hist = adam(init(10, seed=0), X, y, "linear")
slope_net = (p["W1"] @ p["W2"])[0, 0]                  # sum_h w_h v_h
icpt_net  = p["b1"] @ p["W2"][:, 0] + p["b2"][0]       # sum_h b_h v_h + c
print(f"linear net: slope = {slope_net:.4f}, intercept = {icpt_net:+.4f}, MSE = {hist[-1]:.4f}")
print("The 31 parameters collapsed into ONE slope and ONE intercept.")

## 3. Is our backpropagation right?
Same habit as Post 1: compare the hand-written gradient with a central finite difference.

In [ ]:
def finite_diff_check(act, H=5, h=1e-6):
    p = init(H, seed=0)
    _, g = loss_and_grad(p, X, y, act)
    worst = 0.0
    for k in p:
        flat = p[k].reshape(-1)                      # view into the parameter array
        for i in range(flat.size):
            old = flat[i]
            flat[i] = old + h; Lp = loss_and_grad(p, X, y, act)[0]
            flat[i] = old - h; Lm = loss_and_grad(p, X, y, act)[0]
            flat[i] = old
            worst = max(worst, abs((Lp - Lm) / (2*h) - g[k].reshape(-1)[i]))
    return worst

for act in ("tanh", "sigmoid", "relu"):
    print(f"{act:>8}: largest |backprop - finite difference| = {finite_diff_check(act):.1e}")

## 4. Train a tanh network
10 hidden units, Adam with learning rate 0.01, 4000 steps. First the **cubic from Post 1** as a baseline.

In [ ]:
# baseline: the best cubic (no ridge penalty here, plain MSE)
c = np.polynomial.polynomial.polyfit(x / np.pi, y, 3)
cubic = lambda xx: np.polynomial.polynomial.polyval(xx / np.pi, c)
mse_cubic = np.mean((cubic(x) - y)**2)
print(f"best cubic: MSE = {mse_cubic:.2e}")

p_tanh, h_tanh = adam(init(10, seed=0), X, y, "tanh")
print()
print(f"{'updates':>8} {'loss':>12}")
for k in (0, 10, 100, 500, 1000, 2000, 4000):
    print(f"{k:>8} {h_tanh[k]:>12.3e}")

err = np.abs(forward(p_tanh, X, "tanh")[2][:, 0] - y)
print(f"\nfinal MSE = {h_tanh[-1]:.2e}  ({mse_cubic / h_tanh[-1]:.0f}x lower than the cubic)")
print(f"largest error anywhere on the grid = {err.max():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(x, y, lw=3, alpha=0.4, label="sin(x)")
plt.plot(x, cubic(x), "--", label="best cubic")
plt.plot(x, forward(p_tanh, X, "tanh")[2][:, 0], label="tanh network, H=10")
plt.legend(); plt.grid(alpha=0.3); plt.xlabel("x"); plt.title("Cubic vs neural network"); plt.show()

plt.figure(figsize=(6.5, 3.8))
plt.semilogy(h_tanh); plt.xlabel("update"); plt.ylabel("loss"); plt.grid(alpha=0.3)
plt.title("Learning curve"); plt.show()

## 5. What did the network build?
The prediction is a sum of ten scaled S-shaped curves. Plot each term $v_h\tanh(w_hx+b_h)$ and their sum.

In [ ]:
Z, A, yhat = forward(p_tanh, X, "tanh")
contrib = A * p_tanh["W2"][:, 0]                    # (N, H): v_h * tanh(w_h x + b_h)

plt.figure(figsize=(8, 4.2))
for h in range(contrib.shape[1]):
    plt.plot(x, contrib[:, h], lw=1, alpha=0.7)
plt.plot(x, yhat[:, 0], "k", lw=2.5, label="sum + c  (the prediction)")
plt.legend(); plt.grid(alpha=0.3); plt.xlabel("x")
plt.title("Ten tanh 'building blocks' and their sum"); plt.show()

## 6. Compare activation functions
Five random starts each (seeds 0 to 4). We report the average **training** error, the error on **fresh points inside** $[-\pi,\pi]$, and the error **outside** the range, on $[-2\pi,2\pi]$.
**Stop and predict:** which activation wins inside the range? Which one fails completely?

In [ ]:
rows = {}
models = {}
for act in ("linear", "tanh", "sigmoid", "relu"):
    res = []
    for seed in range(5):
        p, h = adam(init(10, seed), X, y, act)
        res.append((h[-1], mse(p, x_test, act), mse(p, x_far, act)))
        models[(act, seed)] = p
    rows[act] = np.mean(res, axis=0)

print(f"{'activation':>10} {'train MSE':>12} {'test MSE':>12} {'outside range':>14}")
for act, (tr, te, far) in rows.items():
    print(f"{act:>10} {tr:>12.2e} {te:>12.2e} {far:>14.3f}")
print(f"{'cubic':>10} {mse_cubic:>12.2e} {'':>12} {np.mean((cubic(x_far) - np.sin(x_far))**2):>14.3f}")
print(f"{'predict 0':>10} {'':>12} {'':>12} {np.mean(np.sin(x_far)**2):>14.3f}")

## 7. How many hidden units?
Mean training MSE over five seeds, same 4000 Adam steps for every size. (This cell takes about 30 seconds.)

In [ ]:
sizes = [1, 2, 3, 5, 10, 20, 50]
sweep = {}
for act in ("tanh", "relu"):
    for H in sizes:
        sweep[(act, H)] = np.mean([adam(init(H, s), X, y, act)[1][-1] for s in range(5)])

print(f"{'H':>4} {'tanh':>12} {'relu':>12}")
for H in sizes:
    print(f"{H:>4} {sweep[('tanh', H)]:>12.2e} {sweep[('relu', H)]:>12.2e}")

## 8. ReLU networks are piecewise linear
Each ReLU unit bends the curve at one point, $x_h=-b_h/w_h$. Units whose bend lies outside the data range add no bend where we need it.

In [ ]:
p_relu = models[("relu", 0)]
kinks = -p_relu["b1"] / p_relu["W1"][0]
inside = (kinks > -np.pi) & (kinks < np.pi)
yhat_relu = forward(p_relu, X, "relu")[2][:, 0]
print("kink locations:", np.round(np.sort(kinks), 2))
print(f"{inside.sum()} of {len(kinks)} kinks lie inside [-pi, pi]")
print(f"largest error = {np.abs(yhat_relu - y).max():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(x, y, lw=3, alpha=0.4, label="sin(x)")
plt.plot(x, yhat_relu, label="ReLU network, H=10")
for k in kinks[inside]:
    plt.axvline(k, color="gray", ls=":", lw=0.8)
plt.legend(); plt.grid(alpha=0.3); plt.xlabel("x"); plt.title("Straight pieces joined at the kinks"); plt.show()

## 9. Outside the training range
**Stop and predict:** what will each network do for $x>\pi$?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(x_far, np.sin(x_far), lw=3, alpha=0.35, label="sin(x)")
ax.plot(x_far, cubic(x_far), label="cubic")
ax.plot(x_far, forward(models[("tanh", 0)], x_far[:, None], "tanh")[2][:, 0], label="tanh net")
ax.plot(x_far, forward(models[("relu", 0)], x_far[:, None], "relu")[2][:, 0], label="ReLU net")
ax.axvspan(-np.pi, np.pi, color="gold", alpha=0.12, label="training range")
ax.set_ylim(-3, 3); ax.legend(ncol=3); ax.grid(alpha=0.3); ax.set_xlabel("x")
ax.set_title("Networks interpolate; they do not know sine is periodic"); plt.show()

## 10. Cross-check with PyTorch autograd
The same network, but PyTorch computes the gradient. (If PyTorch is missing the cell just says so.)

In [ ]:
try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
    print("PyTorch not installed; skipping.")

if HAVE_TORCH:
    DT = torch.float64
    p0 = init(10, seed=0)
    Xt, yt = torch.as_tensor(X, dtype=DT), torch.as_tensor(y, dtype=DT)
    P = {k: torch.nn.Parameter(torch.as_tensor(v.copy(), dtype=DT)) for k, v in p0.items()}

    def loss_torch():
        yh = torch.tanh(Xt @ P["W1"] + P["b1"]) @ P["W2"] + P["b2"]
        return torch.mean((yh[:, 0] - yt) ** 2)

    # 1) gradient at the starting point: autograd vs our backprop
    loss_torch().backward()
    _, g_np = loss_and_grad(p0, X, y, "tanh")
    worst = max(np.max(np.abs(P[k].grad.numpy() - g_np[k])) for k in P)
    print(f"autograd vs hand-written backprop: largest difference = {worst:.1e}")

    # 2) train with torch.optim.Adam, same settings
    opt = torch.optim.Adam(P.values(), lr=0.01)
    for _ in range(4000):
        opt.zero_grad(set_to_none=True)
        loss_torch().backward()
        opt.step()
    print(f"PyTorch final MSE = {loss_torch().item():.2e}   (NumPy: {h_tanh[-1]:.2e})")

## 11. Your turn
**Exercise 1 (linear collapse).** For $W_1=[1,2]$, $b_1=[0,1]$, $W_2=[3,-1]^\mathsf T$, $b_2=0.5$ with linear activations, work out the slope and intercept by hand, then check with the code below.

**Exercise 2 (derivatives).** Compute $\sigma'(0)$, $\tanh'(0)$ and $\tanh'(3)$. Which activation passes the most gradient back through a unit that is far from zero?

**Exercise 3 (backprop by hand).** One data point $x=1,\ y=1$, one tanh unit with $w=0.5,\ b=0,\ v=2,\ c=0$. Compute all four gradients by hand, then verify them with the starter code.

**Exercise 4 (dead units).** For the ReLU networks of seeds 0 to 4, how many of the 10 kinks lie inside $[-\pi,\pi]$? Does the number of useful kinks predict the final error?

**Exercise 5 (initialisation).** Set every weight of a tanh network to the *same* value and train it. What happens to the ten hidden units, and why?

In [ ]:
# Exercise 1 and 3 starter: a tiny network by hand
def tiny(W1, b1, W2, b2, xs, ys, act="linear"):
    p = dict(W1=np.array([W1], float), b1=np.array(b1, float),
             W2=np.array(W2, float)[:, None], b2=np.array([b2], float))
    L, g = loss_and_grad(p, np.array(xs, float)[:, None], np.array(ys, float), act)
    return p, L, g

# Exercise 1: probe the collapsed line at x = 0 and x = 1
p, _, _ = tiny([1, 2], [0, 1], [3, -1], 0.5, [0.0, 1.0], [0.0, 0.0])
print("prediction at x=0 and x=1:", forward(p, np.array([[0.0], [1.0]]), "linear")[2][:, 0])

# Exercise 3: gradients for one data point and one tanh unit
_, L, g = tiny([0.5], [0.0], [2.0], 0.0, [1.0], [1.0], act="tanh")
print("loss:", L)
for k, v in g.items():
    print(k, v.ravel())

In [ ]:
# Exercise 4 starter
for seed in range(5):
    p = models[("relu", seed)]
    k = -p["b1"] / p["W1"][0]
    n_in = int(((k > -np.pi) & (k < np.pi)).sum())
    print(f"seed {seed}: kinks inside = {n_in:>2}   train MSE = {mse(p, x, 'relu'):.2e}")

In [ ]:
# Exercise 5 starter: identical hidden units
p_same = dict(W1=np.full((1, 10), 0.7), b1=np.full(10, 0.1),
              W2=np.full((10, 1), 0.3), b2=np.zeros(1))
p_same, h_same = adam(p_same, X, y, "tanh", steps=2000)
print("final MSE:", h_same[-1])
print("hidden-unit weights w_h:", np.round(p_same["W1"][0], 6))
# TODO: compare with init(10, seed=0) and explain

## 12. Self-check
If everything ran, these assertions confirm that you reproduced the numbers in the post.

In [ ]:
assert abs(mse_line - 0.1996) < 1e-3 and abs(slope_net - slope_line) < 1e-3
assert max(finite_diff_check(a) for a in ("tanh", "sigmoid", "relu")) < 1e-7
assert abs(mse_cubic - 0.00457) < 1e-4
assert h_tanh[-1] < 1e-5 and 1.4 < h_tanh[0] < 1.7
assert rows["tanh"][0] < 2e-5 and rows["relu"][0] > 1e-4
assert rows["linear"][0] > 0.19 and rows["tanh"][2] < rows["relu"][2] < rows["linear"][2]
print("All checks passed: you reproduced the numbers in the post.")